# Feature Engineering — Pharma Sales Forecasting
---
**Pipeline:** Raw monthly sales → cleaned, feature-rich, scaled dataset ready for ML.

Steps:
1. Load & clean data
2. Handle missing / anomalous values
3. Create time-based features
4. Lag features for time-series
5. Rolling-window statistics
6. Cross-drug aggregate features
7. Encode categoricals
8. Feature scaling
9. Time-aware train-test split

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Config
DRUG_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
TARGET_COL = "N02BE"  # Highest-volume drug — primary forecasting target
LAG_PERIODS = [1, 2, 3, 6, 12]
ROLLING_WINDOWS = [3, 6, 12]
TEST_FRACTION = 0.20

OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup complete ✓")

Setup complete ✓


## 2. Load Raw Data

In [2]:
df = pd.read_csv("salesmonthly.csv", parse_dates=["datum"])
df.sort_values("datum", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Loaded {len(df)} rows × {df.shape[1]} columns")
print(f"Date range: {df['datum'].min().date()} → {df['datum'].max().date()}")
df.head()

Loaded 70 rows × 9 columns
Date range: 2014-01-31 → 2019-10-31


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-31,127.69,99.090,152.100,878.030,354.0,50.0,112.0,48.2
1,2014-02-28,133.32,126.050,177.000,1001.900,347.0,31.0,122.0,36.2
2,2014-03-31,137.44,92.950,147.655,779.275,232.0,20.0,112.0,85.4
3,2014-04-30,113.10,89.475,130.900,698.500,209.0,18.0,97.0,73.7
4,2014-05-31,101.79,119.933,132.100,628.780,270.0,23.0,107.0,123.7


## 3. Handle Missing & Anomalous Values

In [3]:
# Detect anomalous rows (Jan 2017 has all-zero sales — data gap)
zero_mask = df[DRUG_COLS].sum(axis=1) <= 1.0
print(f"Near-zero rows found: {zero_mask.sum()}")
print(df[zero_mask][["datum"] + DRUG_COLS])

# Replace with NaN and interpolate linearly
df.loc[zero_mask, DRUG_COLS] = np.nan

for col in DRUG_COLS:
    df[col] = df[col].interpolate(method="linear")

# Edge-fill any remaining NaNs
df[DRUG_COLS] = df[DRUG_COLS].bfill().ffill()

print(f"\nMissing values after imputation: {df[DRUG_COLS].isnull().sum().sum()}")

Near-zero rows found: 1
        datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C  R03  R06
36 2017-01-31    0.0    0.0    0.0    0.0   1.0   0.0  0.0  0.0

Missing values after imputation: 0


## 4. Time-Based Features

In [4]:
# Extract calendar features from date
dt = df["datum"].dt

df["year"]         = dt.year
df["month"]        = dt.month
df["quarter"]      = dt.quarter
df["week_of_year"] = dt.isocalendar().week.astype(int)
df["day_of_year"]  = dt.dayofyear

# Season mapping (Northern Hemisphere)
season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring",  4: "Spring",  5: "Spring",
              6: "Summer",  7: "Summer",  8: "Summer",
              9: "Autumn", 10: "Autumn", 11: "Autumn"}
df["season"] = df["month"].map(season_map)

# Binary flags
df["is_year_start"] = (df["month"] == 1).astype(int)
df["is_year_end"]   = (df["month"] == 12).astype(int)

# Cyclical encoding — so ML sees Jan↔Dec continuity
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

print("Time features created:")
print(df[["datum", "year", "month", "quarter", "season",
          "month_sin", "month_cos"]].head(8))

Time features created:
       datum  year  month  quarter  season     month_sin     month_cos
0 2014-01-31  2014      1        1  Winter  5.000000e-01  8.660254e-01
1 2014-02-28  2014      2        1  Winter  8.660254e-01  5.000000e-01
2 2014-03-31  2014      3        1  Spring  1.000000e+00  6.123234e-17
3 2014-04-30  2014      4        2  Spring  8.660254e-01 -5.000000e-01
4 2014-05-31  2014      5        2  Spring  5.000000e-01 -8.660254e-01
5 2014-06-30  2014      6        2  Summer  1.224647e-16 -1.000000e+00
6 2014-07-31  2014      7        3  Summer -5.000000e-01 -8.660254e-01
7 2014-08-31  2014      8        3  Summer -8.660254e-01 -5.000000e-01


## 5. Lag Features

In [5]:
# Lag features capture autocorrelation at different horizons
for lag in LAG_PERIODS:
    df[f"{TARGET_COL}_lag_{lag}"] = df[TARGET_COL].shift(lag)

lag_cols = [c for c in df.columns if "_lag_" in c]
print(f"Created {len(lag_cols)} lag features: {lag_cols}")
df[["datum", TARGET_COL] + lag_cols].tail(8)

Created 5 lag features: ['N02BE_lag_1', 'N02BE_lag_2', 'N02BE_lag_3', 'N02BE_lag_6', 'N02BE_lag_12']


,datum,N02BE,N02BE_lag_1,N02BE_lag_2,N02BE_lag_3,N02BE_lag_6,N02BE_lag_12
62,2019-03-31,941.050,1001.212,1660.612,1213.950,1058.262,999.123
63,2019-04-30,647.650,941.050,1001.212,1660.612,1129.275,836.037
64,2019-05-31,703.562,647.650,941.050,1001.212,995.150,644.648
65,2019-06-30,610.000,703.562,647.650,941.050,1213.950,584.343
66,2019-07-31,649.800,610.000,703.562,647.650,1660.612,679.350
67,2019-08-31,518.100,649.800,610.000,703.562,1001.212,733.838
68,2019-09-30,984.480,518.100,649.800,610.000,941.050,1058.262
69,2019-10-31,295.150,984.480,518.100,649.800,647.650,1129.275


## 6. Rolling-Window Features

In [6]:
# Rolling mean and std at 3, 6, 12 month windows
for window in ROLLING_WINDOWS:
    df[f"{TARGET_COL}_roll_mean_{window}"] = (
        df[TARGET_COL].rolling(window=window, min_periods=1).mean()
    )
    df[f"{TARGET_COL}_roll_std_{window}"] = (
        df[TARGET_COL].rolling(window=window, min_periods=1).std()
    )

# Expanding (cumulative) mean
df[f"{TARGET_COL}_expanding_mean"] = df[TARGET_COL].expanding(min_periods=1).mean()

# Month-over-month percentage change
df[f"{TARGET_COL}_mom_change"] = df[TARGET_COL].pct_change()

roll_cols = [c for c in df.columns if "roll_" in c or "expanding" in c or "mom_" in c]
print(f"Created {len(roll_cols)} rolling/expanding features")
df[["datum", TARGET_COL] + roll_cols].tail(8)

Created 8 rolling/expanding features


,datum,N02BE,N02BE_roll_mean_3,N02BE_roll_std_3,N02BE_roll_mean_6,N02BE_roll_std_6,N02BE_roll_mean_12,N02BE_roll_std_12,N02BE_expanding_mean,N02BE_mom_change
62,2019-03-31,941.050,1200.958000,399.206984,1156.874833,266.359403,956.477250,298.883821,938.802310,-0.060089
63,2019-04-30,647.650,863.304000,189.168846,1076.604000,339.005722,940.778333,310.506605,934.253055,-0.311779
64,2019-05-31,703.562,764.087333,155.783109,1028.006000,372.284671,945.687833,305.829282,930.703962,0.086331
65,2019-06-30,610.000,653.737333,47.077104,927.347667,393.024332,947.825917,303.151390,925.844811,-0.132983
66,2019-07-31,649.800,654.454000,46.954305,758.879000,168.170510,945.363417,305.640279,921.724739,0.065246
67,2019-08-31,518.100,592.633333,67.545713,678.360333,142.657851,927.385250,324.948700,915.789081,-0.202678
68,2019-09-30,984.480,717.460000,240.439146,685.598667,158.836608,921.236750,322.939014,916.784601,0.900174
69,2019-10-31,295.150,599.243333,351.755800,626.848667,226.471201,851.726333,361.550952,907.904107,-0.700197


## 7. Cross-Drug Aggregate Features

In [7]:
# Portfolio-level demand signals
df["total_sales"]     = df[DRUG_COLS].sum(axis=1)
df["target_share"]    = df[TARGET_COL] / df["total_sales"].replace(0, np.nan)
df["anti_inflam_total"] = df["M01AB"] + df["M01AE"]
df["analgesic_total"]   = df["N02BA"] + df["N02BE"]
df["respiratory_total"] = df["R03"] + df["R06"]

print("Cross-drug features:")
df[["datum", "total_sales", "target_share", "anti_inflam_total",
    "analgesic_total", "respiratory_total"]].tail(5)

Cross-drug features:


,datum,total_sales,target_share,anti_inflam_total,analgesic_total,respiratory_total
65,2019-06-30,1482.407,0.411493,253.167,713.20,298.04
66,2019-07-31,1517.941,0.428080,284.541,742.60,220.20
67,2019-08-31,1377.779,0.376040,270.179,602.30,242.30
68,2019-09-30,1864.387,0.528045,272.507,1077.98,270.10
69,2019-10-31,538.600,0.547995,81.670,315.80,48.13


## 8. Encode Categorical Variables

In [8]:
# One-hot encode season
df = pd.get_dummies(df, columns=["season"], prefix="season", drop_first=False, dtype=int)

season_cols = [c for c in df.columns if c.startswith("season_")]
print(f"Season dummies: {season_cols}")
df[["datum", "month"] + season_cols].head(8)

Season dummies: ['season_Autumn', 'season_Spring', 'season_Summer', 'season_Winter']


,datum,month,season_Autumn,season_Spring,season_Summer,season_Winter
0,2014-01-31,1,0,0,0,1
1,2014-02-28,2,0,0,0,1
2,2014-03-31,3,0,1,0,0
3,2014-04-30,4,0,1,0,0
4,2014-05-31,5,0,1,0,0
5,2014-06-30,6,0,0,1,0
6,2014-07-31,7,0,0,1,0
7,2014-08-31,8,0,0,1,0


## 9. Drop Rows with NaN from Lag/Rolling

In [9]:
# Lag features introduce NaN for the first 12 rows — drop them
initial_len = len(df)
df.dropna(inplace=True)
print(f"Dropped {initial_len - len(df)} rows (lag warm-up period)")
print(f"Remaining: {len(df)} rows ({df['datum'].min().date()} → {df['datum'].max().date()})")

Dropped 12 rows (lag warm-up period)
Remaining: 58 rows (2015-01-31 → 2019-10-31)


## 10. Time-Aware Train-Test Split

In [10]:
# Chronological split — last 20% of time as test set (no shuffling!)
split_idx = int(len(df) * (1 - TEST_FRACTION))
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f"Train: {len(train)} rows ({train['datum'].min().date()} → {train['datum'].max().date()})")
print(f"Test:  {len(test)} rows  ({test['datum'].min().date()} → {test['datum'].max().date()})")

Train: 46 rows (2015-01-31 → 2018-10-31)
Test:  12 rows  (2018-11-30 → 2019-10-31)


## 11. Feature Scaling

In [11]:
# Identify feature columns (everything except target and date)
exclude = {"datum", TARGET_COL}
feature_cols = [c for c in train.columns if c not in exclude]

print(f"Feature count: {len(feature_cols)}")
print(f"Features: {feature_cols}")

Feature count: 38
Features: ['M01AB', 'M01AE', 'N02BA', 'N05B', 'N05C', 'R03', 'R06', 'year', 'month', 'quarter', 'week_of_year', 'day_of_year', 'is_year_start', 'is_year_end', 'month_sin', 'month_cos', 'N02BE_lag_1', 'N02BE_lag_2', 'N02BE_lag_3', 'N02BE_lag_6', 'N02BE_lag_12', 'N02BE_roll_mean_3', 'N02BE_roll_std_3', 'N02BE_roll_mean_6', 'N02BE_roll_std_6', 'N02BE_roll_mean_12', 'N02BE_roll_std_12', 'N02BE_expanding_mean', 'N02BE_mom_change', 'total_sales', 'target_share', 'anti_inflam_total', 'analgesic_total', 'respiratory_total', 'season_Autumn', 'season_Spring', 'season_Summer', 'season_Winter']


In [12]:
# Fit scaler on TRAIN only, transform both sets
scaler = StandardScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols]  = scaler.transform(test[feature_cols])

print("Scaling complete ✓")
print(f"\nTrain feature stats (should be ~0 mean, ~1 std):")
train[feature_cols].describe().loc[["mean", "std"]].round(3)

Scaling complete ✓

Train feature stats (should be ~0 mean, ~1 std):


,M01AB,M01AE,N02BA,N05B,N05C,R03,R06,year,month,quarter,...,N02BE_mom_change,total_sales,target_share,anti_inflam_total,analgesic_total,respiratory_total,season_Autumn,season_Spring,season_Summer,season_Winter
mean,-0.000,-0.000,0.000,0.000,0.000,0.000,0.000,-0.000,0.000,-0.000,...,0.000,-0.000,0.000,0.000,-0.000,0.000,-0.000,-0.000,-0.000,0.000
std,1.011,1.011,1.011,1.011,1.011,1.011,1.011,1.011,1.011,1.011,...,1.011,1.011,1.011,1.011,1.011,1.011,1.011,1.011,1.011,1.011


## 12. Save Processed Data

In [13]:
# Persist everything needed for model training
train.to_csv(OUTPUT_DIR / "train.csv", index=False)
test.to_csv(OUTPUT_DIR / "test.csv", index=False)
joblib.dump(scaler, OUTPUT_DIR / "feature_scaler.joblib")
joblib.dump(feature_cols, OUTPUT_DIR / "feature_columns.joblib")

print(f"✓ Saved train.csv       ({len(train)} rows)")
print(f"✓ Saved test.csv        ({len(test)} rows)")
print(f"✓ Saved feature_scaler.joblib")
print(f"✓ Saved feature_columns.joblib")
print(f"\nOutput directory: {OUTPUT_DIR}/")

✓ Saved train.csv       (46 rows)
✓ Saved test.csv        (12 rows)
✓ Saved feature_scaler.joblib
✓ Saved feature_columns.joblib

Output directory: processed_data/


## Feature Summary

| Category | Features | Count |
|----------|----------|-------|
| Calendar | year, month, quarter, week_of_year, day_of_year, is_year_start, is_year_end | 7 |
| Cyclical | month_sin, month_cos | 2 |
| Lag | lag_1, lag_2, lag_3, lag_6, lag_12 | 5 |
| Rolling | roll_mean_3/6/12, roll_std_3/6/12, expanding_mean, mom_change | 8 |
| Cross-drug | total_sales, target_share, anti_inflam_total, analgesic_total, respiratory_total | 5 |
| Drug columns | M01AB, M01AE, N02BA, N05B, N05C, R03, R06 | 7 |
| Season dummies | season_Autumn, season_Spring, season_Summer, season_Winter | 4 |
| **Total** | | **38** |